# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JustAnn1234/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked Content Actions & Archetype Mapping

We map model predictions ($\hat{y}$) and feature signatures into three distinct content action archetypes:

1. **`PRIORITY_REWRITE` (`HIGH_DEMAND_POSITION_DECAY`):**
   * *Criteria:* Model predicted decay probability $> 0.70$ AND total impressions $\ge 500$ AND average position $> 10.0$.
   * *Playbook Action:* Comprehensive editorial overhaul, intent re-alignment, and structural content refresh.
2. **`LIGHT_REFRESH` (`MODERATE_DECAY_STRIKING_DISTANCE`):**
   * *Criteria:* Model predicted decay probability between $0.40$ and $0.70$ AND average position between $8.0$ and $20.0$.
   * *Playbook Action:* On-page optimization, title/H2 heading updates, internal link additions, and schema markup updates.
3. **`MONITOR` (`STABLE_PERFORMER`):**
   * *Criteria:* Predicted decay probability $< 0.40$.
   * *Playbook Action:* Passive performance tracking; no immediate editorial intervention required.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import json
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata
from huggingface_hub import hf_hub_download, list_repo_files
from sklearn.ensemble import RandomForestClassifier

# 1. Retrieve Hugging Face Read Token securely
hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN')
    print("Successfully retrieved HF_TOKEN from Colab Secrets.")
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    raise ValueError("HF_TOKEN not found! Please set HF_TOKEN in Colab Secrets.")

# 2. Download March 2026 dataset slice locally
repo_id = "FlyRank/internship-warehouse"
repo_files = list_repo_files(repo_id=repo_id, repo_type="dataset", token=hf_token)
mar_files = [f for f in repo_files if "fact_content_daily_performance/month=2026-03" in f]

local_mar_path = hf_hub_download(
    repo_id=repo_id,
    filename=mar_files[0],
    repo_type="dataset",
    token=hf_token
)

con = duckdb.connect()

# Query feature matrix and aggregate metrics
q_playbook = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions) AS total_impressions,
    SUM(f.gsc_clicks) AS total_clicks,
    ROUND(SUM(f.gsc_sum_position) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0), 2) AS feat_avg_position_mar,
    LN(SUM(f.gsc_impressions) + 1) AS feat_log_impressions_mar,
    ROUND(CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks) * 1.0 / SUM(f.gsc_impressions) ELSE 0 END, 4) AS feat_ctr_mar,
    ROUND(COUNT(CASE WHEN f.gsc_impressions > 0 THEN 1 END) * 1.0 / 31.0, 4) AS feat_active_days_ratio_mar,
    ROUND(CASE WHEN SUM(f.ga4_sessions) > 0 THEN SUM(f.sessions_ai) * 1.0 / SUM(f.ga4_sessions) ELSE 0 END, 4) AS feat_ai_session_ratio_mar,
    CASE WHEN (SUM(f.gsc_sum_position) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0)) > 15.0 OR (SUM(f.gsc_impressions) < 50) THEN 1 ELSE 0 END AS target_is_declining
FROM '{local_mar_path}' f
WHERE f.gsc_data_available IS TRUE
GROUP BY f.client_hash_id, f.content_hash_id
HAVING SUM(f.gsc_impressions) >= 100;
"""

df_pb = con.execute(q_playbook).df()

feature_cols = [
    'feat_log_impressions_mar',
    'feat_avg_position_mar',
    'feat_ctr_mar',
    'feat_active_days_ratio_mar',
    'feat_ai_session_ratio_mar'
]

# Train production decision-support Random Forest model
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(df_pb[feature_cols].fillna(0), df_pb['target_is_declining'])

df_pb['model_decay_prob'] = rf.predict_proba(df_pb[feature_cols].fillna(0))[:, 1]

# Apply Playbook Mapping
def assign_action(row):
    prob = row['model_decay_prob']
    pos = row['feat_avg_position_mar']
    imps = row['total_impressions']

    if prob >= 0.70 and imps >= 500 and pos > 10.0:
        return 'PRIORITY_REWRITE', 'HIGH_DEMAND_POSITION_DECAY'
    elif prob >= 0.40 and pos >= 8.0 and pos <= 20.0:
        return 'LIGHT_REFRESH', 'MODERATE_DECAY_STRIKING_DISTANCE'
    else:
        return 'MONITOR', 'STABLE_PERFORMER'

actions_reasons = df_pb.apply(assign_action, axis=1)
df_pb['action_label'] = [a[0] for a in actions_reasons]
df_pb['reason_code'] = [a[1] for a in actions_reasons]

# Sort queue by priority score
df_pb['priority_score'] = (df_pb['model_decay_prob'] * 50) + (np.log1p(df_pb['total_impressions']) * 10)
df_pb = df_pb.sort_values(by='priority_score', ascending=False).reset_index(drop=True)

print("=== ACTION BREAKDOWN IN RANKED QUEUE ===")
print(df_pb['action_label'].value_counts())

Successfully retrieved HF_TOKEN from Colab Secrets.


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

=== ACTION BREAKDOWN IN RANKED QUEUE ===
action_label
MONITOR             82488
PRIORITY_REWRITE    15319
LIGHT_REFRESH        3634
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use & Operational Limits

**Intended Scope:**
* **Primary Users:** Content Strategists, SEO Specialists, and Editorial Lead Managers.
* **Core Application:** Directional triage for resource allocation during monthly or quarterly content planning cycles.

**Operational Boundaries (Where Validity Ends):**
1. **Seasonal Search Shifts:** Model decay flags assume steady user demand; seasonal terms (e.g., holiday guides) will trigger false-positive decay warnings during off-peak months.
2. **Recent Website Migrations:** URL structure changes or domain updates temporarily disrupt ranking positions and impression collection, rendering model predictions invalid during migration windows.
3. **Massive Search Algorithm Updates:** Major core updates that reshape SERP features (e.g., expansion of AI Overviews) require baseline retraining before relying on output scores.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Summary metrics of action distribution for documentation
playbook_summary = df_pb.groupby('action_label').agg(
    n_items=('content_hash_id', 'count'),
    avg_impressions=('total_impressions', 'mean'),
    avg_position=('feat_avg_position_mar', 'mean'),
    avg_decay_prob=('model_decay_prob', 'mean')
).round(2)

print("=== INTENDED USE ACTION ARCHETYPE SUMMARY ===")
print(playbook_summary.to_string())

=== INTENDED USE ACTION ARCHETYPE SUMMARY ===
                  n_items  avg_impressions  avg_position  avg_decay_prob
action_label                                                            
LIGHT_REFRESH        3634           256.03         17.37            0.95
MONITOR             82488          2528.00         11.70            0.17
PRIORITY_REWRITE    15319          4526.13         27.87            0.94


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-in-the-Loop Review Protocol & The No-Go List

**Human Review Checklist (Mandatory before applying updates):**
* **SERP Feature Intent Inspection:** Verify whether lost CTR is caused by position drops or Google introducing a featured snippet or AI Overview block at position 0.
* **Keyword Cannibalization Audit:** Confirm that a declining page isn't losing traffic to a newly published sibling article targeting the same keyword cluster.
* **Conversion / Commercial Value Verification:** Ensure high-impression pages driving low conversions are prioritized below lower-volume high-converting product pages.

---

### The Automation No-Go List (STRICT)

1. **NO Automated Article Publishing or Overwriting:** Machine learning scores must **never** trigger autonomous LLM article overwrites without human editorial sign-off.
2. **NO Automated URL Redirects or Deletions:** High-decay probabilities must **never** trigger automatic 301 redirects, canonical tag changes, or HTTP 410 content deletions.
3. **NO Unreviewed Metadata Changes:** Title tag and meta description rewrites must pass human review to prevent accidental branding or compliance violations.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify top 5 Priority Rewrite Candidates for Human Review
top_human_review = df_pb[df_pb['action_label'] == 'PRIORITY_REWRITE'].head(5)[
    ['content_hash_id', 'total_impressions', 'feat_avg_position_mar', 'model_decay_prob', 'reason_code']
]

print("=== SAMPLE TOP-5 CANDIDATES REQUIRING MANDATORY HUMAN REVIEW ===")
print(top_human_review.to_string(index=False))

=== SAMPLE TOP-5 CANDIDATES REQUIRING MANDATORY HUMAN REVIEW ===
         content_hash_id  total_impressions  feat_avg_position_mar  model_decay_prob                reason_code
content_e8a52cf3d5988c07           244931.0                  15.17          0.973443 HIGH_DEMAND_POSITION_DECAY
content_36e53e9c707674fc           194579.0                  32.79          0.982288 HIGH_DEMAND_POSITION_DECAY
content_3df3f32f3fd58dea           140156.0                  23.59          0.971013 HIGH_DEMAND_POSITION_DECAY
content_bdf60c86117079be           112429.0                  30.78          0.990546 HIGH_DEMAND_POSITION_DECAY
content_82e35c4845e6c391           143907.0                  22.14          0.936026 HIGH_DEMAND_POSITION_DECAY


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Model Drift Monitoring & Retrain Triggers

To prevent recommendation staleness and performance degradation, the content playbook relies on three explicit operational triggers:

1. **Data Drift Trigger:** Retrain model if the monthly mean impression distribution across active pages shifts by $> 25\%$ compared to the March 2026 baseline.
2. **Concept Drift / Precision Drop:** Trigger model re-calibration if human editor audit precision on `PRIORITY_REWRITE` candidates falls below $70\%$ over two consecutive review cycles.
3. **Cadence Trigger:** Mandated quarterly retraining using rolling 90-day performance windows.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create monitoring JSON receipt
monitoring_receipt = {
    "model_name": "FlyRank_Lane2_RandomForest",
    "baseline_window": "2026-03",
    "n_scored_items": len(df_pb),
    "priority_rewrite_count": int((df_pb['action_label'] == 'PRIORITY_REWRITE').sum()),
    "light_refresh_count": int((df_pb['action_label'] == 'LIGHT_REFRESH').sum()),
    "monitor_count": int((df_pb['action_label'] == 'MONITOR').sum()),
    "retrain_triggers": {
        "monthly_cadence_days": 90,
        "impression_drift_threshold": 0.25,
        "min_human_precision": 0.70
    }
}

os.makedirs('work/outputs', exist_ok=True)
with open('work/outputs/monitoring_receipt.json', 'w') as f:
    json.dump(monitoring_receipt, f, indent=2)

print("=== MONITORING RECEIPT CREATED ===")
print(json.dumps(monitoring_receipt, indent=2))

=== MONITORING RECEIPT CREATED ===
{
  "model_name": "FlyRank_Lane2_RandomForest",
  "baseline_window": "2026-03",
  "n_scored_items": 101441,
  "priority_rewrite_count": 15319,
  "light_refresh_count": 3634,
  "monitor_count": 82488,
  "retrain_triggers": {
    "monthly_cadence_days": 90,
    "impression_drift_threshold": 0.25,
    "min_human_precision": 0.7
  }
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Output Generation & Paper Asset Exports

We export the scored content queue, metric receipts, and key visual assets directly to `work/outputs/` and `work/figures/` for seamless inclusion in the final research paper.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Export Ranked Queue CSV to work/outputs/
queue_export_path = 'work/outputs/ranked_content_queue.csv'
df_pb.to_csv(queue_export_path, index=False)
print(f"Ranked Queue exported successfully to: {queue_export_path}")

# 2. Export Figure: Action Distribution Chart
plt.figure(figsize=(8, 5))
action_counts = df_pb['action_label'].value_counts()
colors = ['#d9534f', '#f0ad4e', '#5cb85c']
plt.bar(action_counts.index, action_counts.values, color=colors, edgecolor='black')
plt.title('Content Playbook Action Triage Breakdown (March 2026)', fontsize=12, fontweight='bold')
plt.xlabel('Playbook Action Archetype', fontsize=10)
plt.ylabel('Number of Content Items', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

fig_path = 'work/figures/action_distribution.png'
plt.savefig(fig_path, dpi=300)
plt.close()
print(f"Figure exported successfully to: {fig_path}")

# 3. Export Metrics Summary Receipt JSON
metrics_receipt = {
    "dataset_rows": len(df_pb),
    "mean_impressions": float(df_pb['total_impressions'].mean()),
    "mean_decay_probability": float(df_pb['model_decay_prob'].mean()),
    "action_breakdown": {k: int(v) for k, v in df_pb['action_label'].value_counts().items()}
}

metrics_json_path = 'work/outputs/playbook_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_receipt, f, indent=2)

print(f"Metrics receipt exported successfully to: {metrics_json_path}")

Ranked Queue exported successfully to: work/outputs/ranked_content_queue.csv
Figure exported successfully to: work/figures/action_distribution.png
Metrics receipt exported successfully to: work/outputs/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w07_action_playbook.ipynb` — then submit your repo URL on the card. Done.